In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# https://www.geeksforgeeks.org/nlp/music-genre-classification-using-transformers/

In [1]:
# !pip install transformers
# !pip install accelerate
# !pip install datasets
# !pip install evaluate

In [4]:
from datasets import load_dataset, Audio
import numpy as np
from transformers import pipeline, AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer
import evaluate
import numpy as np # linear algebra
import copy

In [5]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
def tv(ob):
    print()
    print(f"====================== type =================")
    print(type(ob))
    print(f"====================== value ===================")
    print(ob)
    print()

## Loading dataset and Splitting

In [7]:
gtzan_original = load_dataset("sanchit-gandhi/gtzan")

README.md:   0%|          | 0.00/703 [00:00<?, ?B/s]

data/train-00000-of-00003-abaaa5719027ce(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00001-of-00003-40e2de07ad4288(…):   0%|          | 0.00/429M [00:00<?, ?B/s]

data/train-00002-of-00003-6e2eb838540a06(…):   0%|          | 0.00/436M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/999 [00:00<?, ? examples/s]

In [8]:
# train test split
gtzan_original = gtzan_original["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)

In [9]:
gtzan = copy.deepcopy(gtzan_original)

In [10]:
# print(type(gtzan))
# print(gtzan)
tv(gtzan_original)


====================== type =================
<class 'datasets.dataset_dict.DatasetDict'>
====================== value ===================
DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'genre'],
        num_rows: 899
    })
    test: Dataset({
        features: ['file', 'audio', 'genre'],
        num_rows: 100
    })
})



In [11]:
gtzan['train'][0]

{'file': '/home/sanchit/.cache/datasets/downloads/extracted/f729783d70a4541cc4c9d5649655490a9c660280bdbecddfe38a8a806c73f60e/genres/pop/pop.00098.wav',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7d6488a56e40>,
 'genre': 7}

## Data pre-processing

In [12]:
model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
model_id, do_normalize=True, return_attention_mask=True
)

sampling_rate = feature_extractor.sampling_rate
gtzan = gtzan.cast_column("audio", Audio(sampling_rate=sampling_rate))
sample = gtzan["train"][0]["audio"]
inputs = feature_extractor(
    sample["array"],
    sampling_rate=sample["sampling_rate"]
)
max_duration = 20.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=25,
    num_proc=1,
    )

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
# Renamed the 'genre' column to 'label'
gtzan_encoded = gtzan_encoded.rename_column("genre", "label")
id2label_fn = gtzan["train"].features["genre"].int2str
id2label = {
    str(i): id2label_fn(i)
    for i in range(len(gtzan_encoded["train"].features["label"].names))
}
label2id = {v: k for k, v in id2label.items()}

In [14]:
num_labels = len(id2label)
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

model_name = model_id.split("/")[-1]
batch_size = 2
gradient_accumulation_steps = 1
num_train_epochs = 5

training_args = TrainingArguments(
    f"{model_name}-Music classification Finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/94.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/49 [00:00<?, ?it/s]

HubertForSequenceClassification LOAD REPORT from: ntu-spml/distilhubert
Key               | Status  | 
------------------+---------+-
projector.bias    | MISSING | 
classifier.bias   | MISSING | 
projector.weight  | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

trainer = Trainer(
    model,
    training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.453564,1.343118,0.660000
2,1.150330,0.829254,0.760000
3,0.124594,0.637919,0.810000
4,0.076329,0.788217,0.790000
5,0.063010,0.652925,0.840000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2250, training_loss=0.8569151950412326, metrics={'train_runtime': 1859.8578, 'train_samples_per_second': 2.417, 'train_steps_per_second': 1.21, 'total_flos': 2.044662758208e+17, 'train_loss': 0.8569151950412326, 'epoch': 5.0})

In [16]:
# Save the model and feature extractor
model.save_pretrained("/content/Saved Model")
feature_extractor.save_pretrained("/content/Saved Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/Saved Model/preprocessor_config.json']

In [18]:
# Load the model and feature extractor
loaded_model = AutoModelForAudioClassification.from_pretrained("/content/Saved Model")
loaded_feature_extractor = AutoFeatureExtractor.from_pretrained("/content/Saved Model")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

In [21]:
from transformers import pipeline, AutoFeatureExtractor

pipe = pipeline("audio-classification", model=loaded_model,
                feature_extractor=loaded_feature_extractor)

def classify_audio(filepath):
    preds = pipe(filepath)
    outputs = {}
    for p in preds:
        outputs[p["label"]] = p["score"]
    return outputs

# Provide the input file path
input_file_path = input('Input:')

# Classify the audio file
output = classify_audio(input_file_path)

# Print the output genre
print("Predicted Genre:")
max_key = max(output, key=output.get)

print("The predicted genre is:", max_key)
print("The prediction score is:", output[max_key])

Input:/content/song0003.wav
Predicted Genre:
The predicted genre is: disco
The prediction score is: 0.6592977046966553


In [22]:
# zip the folder so that I can download it and use it in kaggle.
!zip -r /content/modelAndFeatureExtractor.zip /content/'Saved Model'

  adding: content/Saved Model/ (stored 0%)
  adding: content/Saved Model/model.safetensors (deflated 7%)
  adding: content/Saved Model/preprocessor_config.json (deflated 35%)
  adding: content/Saved Model/config.json (deflated 63%)
